# 01 — Exploratory Data Analysis: NIH Chest X-ray14

This notebook explores the dataset before training:
- Dataset size and structure
- Label distribution (class imbalance analysis)
- Sample images per pathology
- Multi-label co-occurrence patterns
- Train/Val/Test split statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Load labels
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "labels.csv")
print(f"Total samples: {len(df):,}")
print(f"Splits: {df['split'].value_counts().to_dict()}")
df.head()

In [ ]:
# Label distribution
from src.data.dataset import PATHOLOGY_LABELS

label_counts = Counter()
for labels_str in df["labels"]:
    if pd.isna(labels_str) or labels_str == "No Finding":
        label_counts["No Finding"] += 1
    else:
        for label in labels_str.split("|"):
            label_counts[label.strip()] += 1

fig, ax = plt.subplots(figsize=(12, 6))
labels, counts = zip(*sorted(label_counts.items(), key=lambda x: -x[1]))
ax.barh(labels, counts, color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Label Distribution — NIH Chest X-ray14")
plt.tight_layout()
plt.show()

In [ ]:
# Binary distribution: Normal vs Abnormal
df["is_abnormal"] = df["labels"].apply(lambda x: 0 if pd.isna(x) or x == "No Finding" else 1)
print(f"Normal: {(df['is_abnormal'] == 0).sum():,}")
print(f"Abnormal: {(df['is_abnormal'] == 1).sum():,}")
print(f"Abnormal ratio: {df['is_abnormal'].mean():.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
df["is_abnormal"].value_counts().plot.pie(labels=["Normal", "Abnormal"], autopct="%1.1f%%", ax=ax)
ax.set_title("Normal vs Abnormal Distribution")
ax.set_ylabel("")
plt.show()

In [ ]:
# Multi-label co-occurrence matrix
cooccurrence = np.zeros((len(PATHOLOGY_LABELS), len(PATHOLOGY_LABELS)), dtype=int)

for labels_str in df["labels"]:
    if pd.isna(labels_str) or labels_str == "No Finding":
        continue
    present = [PATHOLOGY_LABELS.index(l.strip()) for l in labels_str.split("|") if l.strip() in PATHOLOGY_LABELS]
    for i in present:
        for j in present:
            cooccurrence[i][j] += 1

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cooccurrence, xticklabels=PATHOLOGY_LABELS, yticklabels=PATHOLOGY_LABELS,
            cmap="YlOrRd", annot=True, fmt="d", ax=ax)
ax.set_title("Pathology Co-occurrence Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Sample images
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
sample_df = df.sample(10, random_state=42)

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    img_path = DATA_DIR / row["image_path"]
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img, cmap="gray")
    ax.set_title(str(row["labels"])[:30], fontsize=8)
    ax.axis("off")

plt.suptitle("Sample Chest X-ray Images", fontsize=14)
plt.tight_layout()
plt.show()